# Tau-bench：用状态变化而不是漂亮回复评估工具 Agent

**面试问题：有状态工具 Agent 为什么不能只做答案字符串匹配，评测环境怎样实现？**

## 回答主线

1. 工具 Agent 的正确性存在于环境终态、调用参数和策略约束，而不只在最终自然语言。
2. 每个评测任务需要独立初始数据库、用户目标、允许动作、期望状态和不可违反条件。
3. 执行器应逐步应用工具并记录返回值、状态差异与策略违规。
4. 评分同时检查任务完成、过程合规和最终答复，防止“嘴上说完成但数据库没变”。
5. Episode 之间必须深拷贝重置环境，否则前一个样本会污染后一个样本。
6. Tau-bench 风格评估的价值是复现多轮真实操作，而不是得到一个脱离轨迹的单分数。

## 真实案例

五个电商客服任务覆盖取消订单、修改地址、重复退款、已发货限制和仅查询状态。候选 Agent 有的回复正确但没调用工具，有的真正改对状态，有的违反政策。我们实现最小状态机、工具执行账本和终态评分。教学实验使用可读的小数据解释机制，结果不能外推为线上收益。

### 输入预览：五个任务与初始订单

In [1]:
from copy import deepcopy  # 导入深拷贝以隔离每个评测 Episode。

orders = {  # 构造五个脱敏订单的初始数据库。
    "A100": {"status": "paid", "address": "上海市静安区", "refunded": False},  # 可取消的已支付订单。
    "A101": {"status": "paid", "address": "北京市海淀区", "refunded": False},  # 可修改地址的未发货订单。
    "A102": {"status": "cancelled", "address": "杭州市西湖区", "refunded": True},  # 已完成退款的订单。
    "A103": {"status": "shipped", "address": "深圳市南山区", "refunded": False},  # 不允许修改地址的已发货订单。
    "A104": {"status": "paid", "address": "成都市高新区", "refunded": False},  # 只需查询状态的订单。
}  # 完成初始环境。
tasks = [  # 定义用户目标、期望终态和候选 Agent 计划。
    {"id": "E1", "order": "A100", "goal": "取消订单", "expected": {"status": "cancelled", "refunded": True}, "plan": [("cancel_order", {}), ("refund", {})], "reply": "已取消并退款"},  # 构造正确完成任务的轨迹。
    {"id": "E2", "order": "A101", "goal": "地址改到朝阳区", "expected": {"address": "北京市朝阳区"}, "plan": [], "reply": "地址已修改为北京市朝阳区"},  # 构造只说不做的欺骗性回复。
    {"id": "E3", "order": "A102", "goal": "确认退款", "expected": {"refunded": True}, "plan": [("refund", {})], "reply": "退款成功"},  # 构造重复退款调用。
    {"id": "E4", "order": "A103", "goal": "地址改到福田区", "expected": {"address": "深圳市福田区"}, "plan": [("change_address", {"address": "深圳市福田区"})], "reply": "地址已修改"},  # 构造违反已发货限制的尝试。
    {"id": "E5", "order": "A104", "goal": "查询状态", "expected": {"status": "paid"}, "plan": [("get_order", {})], "reply": "订单待发货"},  # 构造只读查询任务。
]  # 完成五个评测样本。
print("任务  订单   用户目标             候选计划")  # 输出样本预览表头。
for task in tasks:  # 逐任务展示目标和工具序列。
    print(f"{task['id']}   {task['order']}  {task['goal']:<16} {[name for name, _ in task['plan']]}")  # 展示任务差异。

任务  订单   用户目标             候选计划
E1   A100  取消订单             ['cancel_order', 'refund']
E2   A101  地址改到朝阳区          []
E3   A102  确认退款             ['refund']
E4   A103  地址改到福田区          ['change_address']
E5   A104  查询状态             ['get_order']


## Baseline 基线：只匹配最终回复关键词

In [2]:
keyword_by_goal = {"取消订单": "取消", "地址改到朝阳区": "修改", "确认退款": "退款", "地址改到福田区": "修改", "查询状态": "待发货"}  # 为每种目标设置朴素关键词。
baseline_rows = []  # 收集字符串匹配结果。
for task in tasks:  # 遍历五个候选回复。
    keyword = keyword_by_goal[task["goal"]]  # 读取当前目标的期望关键词。
    passed = keyword in task["reply"]  # 只看回复是否包含关键词。
    baseline_rows.append({"id": task["id"], "passed": passed, "reply": task["reply"]})  # 保存基线判断。
print("任务  字符串通过  最终回复")  # 输出基线结果表头。
for row in baseline_rows:  # 逐任务展示匹配结论。
    print(f"{row['id']}    {str(row['passed']):<8} {row['reply']}")  # 展示五条回复全部看似合理。
print(f"字符串匹配通过率={sum(row['passed'] for row in baseline_rows) / len(baseline_rows):.0%}")  # 暴露无法识别未执行和违规的问题。

任务  字符串通过  最终回复
E1    True     已取消并退款
E2    True     地址已修改为北京市朝阳区
E3    True     退款成功
E4    True     地址已修改
E5    True     订单待发货
字符串匹配通过率=100%


### 核心实现：工具状态机、状态差异与策略违规

In [3]:
def apply_tool(database, order_id, tool_name, arguments):  # 在隔离数据库上执行一个工具动作。
    order = database[order_id]  # 获取当前订单的可变状态。
    before = deepcopy(order)  # 保存动作前状态用于生成 diff。
    violation = None  # 默认当前动作没有策略违规。
    if tool_name == "get_order":  # 处理只读订单查询。
        result = deepcopy(order)  # 返回订单快照而不修改状态。
    elif tool_name == "cancel_order":  # 处理取消订单。
        if order["status"] != "paid":  # 只允许取消尚未发货的已支付订单。
            violation = "only-paid-order-can-cancel"  # 记录策略违规原因。
            result = {"ok": False}  # 返回拒绝结果。
        else:  # 当前订单满足取消条件。
            order["status"] = "cancelled"  # 修改真实环境状态。
            result = {"ok": True}  # 返回成功结果。
    elif tool_name == "refund":  # 处理退款工具。
        if order["refunded"]:  # 拒绝重复退款以保持幂等。
            violation = "duplicate-refund"  # 记录重复副作用。
            result = {"ok": False}  # 返回拒绝结果。
        elif order["status"] != "cancelled":  # 退款前要求订单已取消。
            violation = "refund-before-cancel"  # 记录顺序违规。
            result = {"ok": False}  # 返回拒绝结果。
        else:  # 满足退款前置条件。
            order["refunded"] = True  # 修改退款状态。
            result = {"ok": True}  # 返回成功结果。
    elif tool_name == "change_address":  # 处理地址修改。
        if order["status"] == "shipped":  # 已发货订单禁止修改地址。
            violation = "address-change-after-shipping"  # 记录政策违规。
            result = {"ok": False}  # 返回拒绝结果。
        else:  # 未发货订单允许修改。
            order["address"] = arguments["address"]  # 写入经过工具参数传入的新地址。
            result = {"ok": True}  # 返回成功结果。
    else:  # 处理未知工具名称。
        violation = "unknown-tool"  # 记录工具不存在。
        result = {"ok": False}  # 返回失败结果。
    changed = {key: {"before": before[key], "after": order[key]} for key in order if before[key] != order[key]}  # 计算权威状态差异。
    return result, changed, violation  # 返回工具结果、状态 diff 和违规。

def run_episode(task, initial_database):  # 在全新环境中执行一条候选 Agent 轨迹。
    database = deepcopy(initial_database)  # 深拷贝数据库防止样本互相污染。
    ledger = []  # 保存每个工具动作的执行证据。
    for tool_name, arguments in task["plan"]:  # 按 Agent 给出的顺序执行工具。
        result, changed, violation = apply_tool(database, task["order"], tool_name, arguments)  # 调用状态机。
        ledger.append({"tool": tool_name, "result": result, "diff": changed, "violation": violation})  # 记录完整动作账本。
    actual = database[task["order"]]  # 读取 episode 结束后的权威订单状态。
    state_ok = all(actual.get(key) == value for key, value in task["expected"].items())  # 对比期望终态字段。
    policy_ok = all(event["violation"] is None for event in ledger)  # 要求整个过程没有策略违规。
    return {"id": task["id"], "state": actual, "state_ok": state_ok, "policy_ok": policy_ok, "passed": state_ok and policy_ok, "ledger": ledger}  # 返回结构化评分结果。

demo_result = run_episode(tasks[0], orders)  # 执行正确的取消退款样本。
print("E1 工具账本：")  # 输出状态迁移标题。
for event in demo_result["ledger"]:  # 逐动作展示权威变化。
    print(event)  # 显示取消和退款分别修改了哪个字段。
print("E1 终态：", demo_result["state"])  # 展示最终数据库而非只看回复。

E1 工具账本：
{'tool': 'cancel_order', 'result': {'ok': True}, 'diff': {'status': {'before': 'paid', 'after': 'cancelled'}}, 'violation': None}
{'tool': 'refund', 'result': {'ok': True}, 'diff': {'refunded': {'before': False, 'after': True}}, 'violation': None}
E1 终态： {'status': 'cancelled', 'address': '上海市静安区', 'refunded': True}


## 结果解读：字符串、终态与策略三重对照

In [4]:
evaluation_rows = [run_episode(task, orders) for task in tasks]  # 在独立环境中评估全部五个任务。
print("任务  字符串  终态  策略  最终通过  违规")  # 输出多维评分表头。
for baseline, result in zip(baseline_rows, evaluation_rows):  # 对齐字符串基线与状态评估。
    violations = [event["violation"] for event in result["ledger"] if event["violation"] is not None]  # 汇总当前轨迹违规。
    print(f"{result['id']}    {str(baseline['passed']):<5} {str(result['state_ok']):<5} {str(result['policy_ok']):<5} {str(result['passed']):<8} {violations}")  # 展示基线误判。
stateful_pass_rate = sum(result["passed"] for result in evaluation_rows) / len(evaluation_rows)  # 计算严格评估通过率。
print(f"字符串通过率=100%，状态+策略通过率={stateful_pass_rate:.0%}")  # 用同一批样本量化差异。
print("解读：E2 没有工具调用所以数据库未变；E3 重复退款违规；E4 工具正确拒绝已发货改单，因此都不能因回复好听而通过。")  # 解释三个典型失败。

任务  字符串  终态  策略  最终通过  违规
E1    True  True  True  True     []
E2    True  False True  False    []
E3    True  True  False False    ['duplicate-refund']
E4    True  False False False    ['address-change-after-shipping']
E5    True  True  True  True     []
字符串通过率=100%，状态+策略通过率=40%
解读：E2 没有工具调用所以数据库未变；E3 重复退款违规；E4 工具正确拒绝已发货改单，因此都不能因回复好听而通过。


## 失败案例：复用可变数据库污染后续样本

In [5]:
shared_database = deepcopy(orders)  # 创建会被错误复用的共享评测环境。
first_result, first_diff, first_violation = apply_tool(shared_database, "A100", "cancel_order", {})  # 第一个样本修改共享订单状态。
second_result, second_diff, second_violation = apply_tool(shared_database, "A100", "cancel_order", {})  # 第二次相同任务继承已取消状态而失败。
fresh_database = deepcopy(orders)  # 为重试样本重新创建初态。
fresh_result, fresh_diff, fresh_violation = apply_tool(fresh_database, "A100", "cancel_order", {})  # 在隔离环境中相同动作稳定成功。
print(f"共享环境第一次：result={first_result} diff={first_diff} violation={first_violation}")  # 展示第一次正常修改。
print(f"共享环境第二次：result={second_result} diff={second_diff} violation={second_violation}")  # 展示样本污染造成的假失败。
print(f"独立环境重试：result={fresh_result} diff={fresh_diff} violation={fresh_violation}")  # 展示 reset 修正后的确定性结果。
print("修正策略：每个 task 从版本化 fixture 深拷贝启动，结束后保存 diff，严禁跨 Episode 复用可变对象。")  # 总结评测隔离合同。

共享环境第一次：result={'ok': True} diff={'status': {'before': 'paid', 'after': 'cancelled'}} violation=None
共享环境第二次：result={'ok': False} diff={} violation=only-paid-order-can-cancel
独立环境重试：result={'ok': True} diff={'status': {'before': 'paid', 'after': 'cancelled'}} violation=None
修正策略：每个 task 从版本化 fixture 深拷贝启动，结束后保存 diff，严禁跨 Episode 复用可变对象。


### 生产边界与评测报告

In [6]:
report = {"benchmark": "stateful-support-mini", "fixture_version": "orders-r3", "tasks": len(tasks), "passed": sum(result["passed"] for result in evaluation_rows), "dimensions": ["final_state", "policy", "tool_ledger", "reply"]}  # 构造可复现评测报告。
print("评测报告：", report)  # 展示分数之外的版本和维度。
print("生产替换点：真实 tau-bench 还需多轮用户模拟器、数据库事务、并发隔离、随机种子、多次采样、pass^k 和人工错误分类。")  # 明确教学状态机边界。

评测报告： {'benchmark': 'stateful-support-mini', 'fixture_version': 'orders-r3', 'tasks': 5, 'passed': 2, 'dimensions': ['final_state', 'policy', 'tool_ledger', 'reply']}
生产替换点：真实 tau-bench 还需多轮用户模拟器、数据库事务、并发隔离、随机种子、多次采样、pass^k 和人工错误分类。


## 回归测试：最后只保护状态、策略和隔离

In [7]:
assert evaluation_rows[0]["passed"] and evaluation_rows[4]["passed"]  # 验证正确写操作和只读查询能够通过。
assert baseline_rows[1]["passed"] and not evaluation_rows[1]["passed"]  # 验证字符串基线误判只说不做的 Agent。
assert not evaluation_rows[2]["policy_ok"] and not evaluation_rows[3]["policy_ok"]  # 验证重复退款和已发货改单均被策略层识别。
assert second_violation == "only-paid-order-can-cancel" and fresh_violation is None  # 验证共享状态污染和独立重置修正。
assert orders["A100"]["status"] == "paid"  # 验证原始 fixture 从未被任一 Episode 修改。
print("回归测试通过：真实终态、策略违规、字符串误判、Episode 隔离和 fixture 不变性均成立。")  # 用少量断言总结评测合同。

回归测试通过：真实终态、策略违规、字符串误判、Episode 隔离和 fixture 不变性均成立。
